<a href="https://colab.research.google.com/github/AGN-6/03MIAR---Algoritmos-De-Optimizacion--2025-26/blob/main/Trabajo_Pr%C3%A1ctico_Alvaro_Gil_Natividad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmos de optimización - Trabajo Práctico<br>
Nombre y Apellidos: Álvaro Gil Natividad  <br>
Url: https://github.com/AGN-6/03MIAR---Algoritmos-De-Optimizacion--2025-26/blob/main/Trabajo_Pr%C3%A1ctico_Alvaro_Gil_Natividad.ipynb<br>
Google Colab: https://colab.research.google.com/drive/1eQ856sv-hs4PSlc6Ttf2zh0vhuW01UC1?usp=sharing <br>


### PROBLEMA:

>3. Configuración de Tribunales

Descripción del problema:


- Se precisa configurar tribunales de evaluación para un grupo de 15 alumnos que desean
presentar su Trabajo Fin de Máster (TFM).
- Cada tribunal está compuesto por tres profesores, cada uno desempeñando uno de los
siguientes roles: Presidente, Secretario o Vocal.
- Los profesores han indicado su disponibilidad horaria para participar en los tribunales de
15h a 21h durante la semana del 15 al 19 de abril:


- Número de profesores : 10
- Número de tribunales : 15
- Disponibilidad/Roles : https://bit.ly/41QWk8o
   - 1 indica que profesor tiene disponibilidad
   - 0 en caso contrario

- Hay 15 alumnos, por lo que se deben configurar 15 tribunales buscando la configuración
más equilibrada posible en cuanto a la cantidad de tribunales asignados a cada profesor, es
decir, evitando que un profesor tenga muchos tribunales y otros pocos.

- Obviamente ningún profesor puede asistir a dos tribunales a la misma fecha/hora y no
puede ser convocado a un tribunal al que no tiene disponibilidad.






                                        

#Modelo
- ¿Como represento el espacio de soluciones?
- ¿Cual es la función objetivo?
- ¿Como implemento las restricciones?


El problema se modela como un Problema de Satisfacción de Restricciones (CSP). Cada solución en el espacio de búsqueda es una asignación de un vector $S$ de $N$ componentes (donde $N=15$ tribunales). Cada componente $T_i$ del vector tiene la forma de una tupla:$$T_i = (Slot\_Temporal, Presidente, Secretario, Vocal)$$






La función objetivo principal en este modelo es minimizar el número de restricciones violadas hasta encontrar una solución estrictamente factible:$$\min f(x) = \sum Coste\_Violaciones(x) \rightarrow 0$$Secundariamente, al usar un algoritmo con enfoque determinista de primer ajuste (First-Fit), el algoritmo maximiza la ocupación temprana y compacta el calendario naturalmente. Esto logra ubicar los 15 tribunales en los 2 primeros días disponibles, optimizando así el tiempo global del proceso.






Las restricciones se manejan mediante validaciones lógicas y condicionales durante la construcción iterativa de la solución:

1. **Disponibilidad**: se consulta la matriz binaria de datos para asegurar que el valor sea positivo (if disponibilidad[prof][slot] == 1).

2. **Roles**: se valida que el profesor pertenezca al conjunto de roles permitidos para el puesto (P, S o V).

3. **Solapamiento**: se mantiene un registro de estado dinámico (ocupados_por_slot) para garantizar que un profesor no asista a dos tribunales distintos en la misma franja horaria.

4. **Diferencia de identidad**: se fuerza lógicamente que los tres miembros del tribunal sean personas distintas Presidente != Secretario != Vocal.


#Análisis
- ¿Que complejidad tiene el problema?. Orden de complejidad y Contabilizar el espacio de soluciones

1. **Contabilizar el espacio de soluciones** (Fuerza Bruta)

Para entender la magnitud del problema, primero calculamos el tamaño total del espacio de búsqueda $\Omega$ si intentáramos evaluar todas las combinaciones posibles:
- Tribunales a planificar ($N$): 15
- Slots de tiempo totales ($S$): 35 (5 días $\times$ 7 horas)
- Profesores disponibles ($P$): 10

Para un solo tribunal, necesitamos elegir un slot de tiempo y una terna de profesores distintos con roles específicos (Presidente, Secretario, Vocal). Las variaciones posibles de 3 profesores elegidos entre 10 (importando el rol) son $V_{10,3} = 10 \times 9 \times 8 = 720$.
Por tanto, las combinaciones teóricas para un único tribunal son:

$$Combinaciones = S \times V_{10,3} = 35 \times 720 = 25.200$$Para configurar los 15 tribunales de forma independiente, el espacio de soluciones crece de forma exponencial:
$$|\Omega| \approx (25.200)^{15} \approx 6.1 \times 10^{65} \text{ estados posibles}$$
Este número astronómico demuestra que el problema es de naturaleza combinatoria NP-Duro.

2. **Orden de complejidad del Algoritmo Implementado**

Siguiendo la premisa de la optimización, hemos optado por un Algoritmo Determinista (Constructivo Voraz) en lugar de una metaheurística compleja, dado que logra resolver el problema actual de 15 tribunales de forma inmediata y óptima (en apenas 2 días de calendario).

- Si habláramos de un algoritmo determinista exacto (Fuerza Bruta): su complejidad es exponencial $O((S \times P^3)^N)$. Dejaría de tener sentido práctico a partir de $N>3$ o $4$ tribunales, ya que el tiempo de cómputo pasaría de milisegundos a años debido a la explosión combinatoria.

- Para nuestro algoritmo determinista Voraz (Greedy): el tiempo de ejecución es polinómico $O(N \cdot S \cdot P^3)$, por lo que es rapidísimo computacionalmente.




3. **Análisis de Escalabilidad** (Límites del enfoque determinista)



El límite de este enfoque voraz no es el tiempo, sino la calidad de la solución al aumentar la densidad de tribunales. Al ser un algoritmo "miope" que no retrocede en sus decisiones (no hace backtracking), si tuviéramos que organizar, por ejemplo, 30 tribunales en los 35 slots disponibles, el algoritmo Voraz tomaría decisiones subóptimas al principio que terminarían bloqueando recursos clave (profesores muy demandados).

Llegado a ese punto de saturación (alta densidad de restricciones), la técnica determinista fallaría al encontrar huecos factibles, y sería obligatorio saltar a Metaheurísticas (como Algoritmos Genéticos o Recocido Simulado) para poder desatascar y reordenar el calendario global.

#Diseño
- ¿Que técnica utilizo? ¿Por qué?

La elección de este algoritmo se fundamenta en cuatro razones clave de optimización:

1. **Eficiencia Computacional frente a la Explosión Combinatoria**:

Al tratarse de un problema NP-Duro con un espacio de búsqueda de $\approx 10^{65}$ estados, los métodos exactos (fuerza bruta) son inviables. El algoritmo voraz construye la solución iterativamente asignando variables de una en una sin realizar backtracking (no deshace decisiones), lo que reduce drásticamente el tiempo de ejecución a milisegundos.

2. **Preferencia por el Determinismo**:

Tal y como dictan las buenas prácticas en la resolución de algoritmos de optimización, siempre que una técnica determinista sea capaz de resolver el problema de forma factible, será preferida frente a metaheurísticas estocásticas (como los Algoritmos Genéticos). El determinismo garantiza una reproducibilidad del 100% en los resultados con el menor coste computacional posible.

3. **Compactación Natural del Calendario**:

Al aplicar la regla First-Fit sobre un espacio de tiempo ordenado cronológicamente, el algoritmo no solo encuentra una solución válida, sino que maximiza la ocupación temprana. Esto produce un horario altamente compactado como "efecto secundario" positivo, logrando ubicar los 15 tribunales en apenas los dos primeros días, optimizando el tiempo logístico.

4. **Baja Densidad del Problema Actual**:

El mayor riesgo de un algoritmo voraz es quedarse atrapado en "óptimos locales" o caminos sin salida si toma decisiones subóptimas al principio. Sin embargo, dado que el problema actual requiere ubicar $N=15$ tribunales en un holgado margen de $35$ slots posibles, la densidad de restricciones permite que el enfoque voraz alcance una solución globalmente factible sin atascarse, haciendo innecesario el salto hacia técnicas más complejas de búsqueda en entornos o metaheurísticas.

In [ ]:
import pandas as pd

# ==========================================
# 1. DATOS INTEGRADOS
# ==========================================

# -- Creación del Espacio Temporal (Slots) --
# Definimos los días y horas exactos extraídos del Excel.
dias_base = [15, 16, 17, 18, 19]
horas_base = [15, 16, 17, 18, 19, 20, 21]

# Creamos una lista de diccionarios. Cada diccionario es una franja horaria única (Slot).
# Total: 5 días * 7 horas = 35 slots temporales posibles.
slots = [{'dia': d, 'hora': h, 'id': i} for i, d in enumerate(dias_base) for h in horas_base]


# -- Restricción de Disponibilidad (Matriz Binaria) --
# 1 = Disponible en ese slot temporal.
# 0 = No disponible (Ocupado en otras tareas o fuera de horario).
# Cada lista tiene exactamente 35 elementos, correspondientes a los 35 slots generados arriba.
availability_db = {
    'RRD': [0,1,1,1,0,1,1, 1,0,1,1,1,1,1, 1,1,0,0,1,0,1, 0,1,0,1,1,1,1, 1,1,1,1,1,0,0],
    'QYV': [1,1,1,1,0,0,0, 0,1,1,1,1,0,0, 1,0,0,1,1,1,0, 1,1,1,1,1,1,1, 1,1,1,1,1,1,1],
    'LHL': [0,0,1,1,0,1,1, 1,1,1,0,0,1,1, 1,1,1,1,1,1,1, 1,0,1,1,1,0,1, 0,1,1,0,1,0,1],
    'HLC': [1,0,1,0,1,1,0, 1,0,0,1,1,1,1, 0,0,1,1,1,1,1, 1,0,1,1,0,1,1, 1,1,1,1,1,1,0],
    'MSB': [1,1,0,1,0,1,1, 1,1,1,0,1,1,1, 1,0,1,1,0,1,1, 0,1,1,1,0,1,1, 1,0,1,1,1,1,0],
    'PMQ': [1,1,1,1,1,0,0, 1,1,1,1,1,1,1, 1,1,0,0,1,1,1, 1,1,1,0,0,1,1, 1,1,1,0,1,0,1],
    'QWF': [0,1,1,1,1,1,1, 1,1,0,1,1,0,1, 0,0,1,1,0,0,1, 1,0,0,0,0,1,1, 1,1,1,1,1,0,1],
    'EBB': [1,1,1,1,1,0,0, 1,1,0,1,1,1,0, 1,1,1,0,0,1,1, 0,1,1,1,1,1,1, 0,1,1,1,0,1,0],
    'IOE': [1,0,1,1,0,1,0, 0,1,1,1,1,1,1, 1,1,0,0,0,1,1, 1,1,1,1,1,1,1, 0,1,0,1,1,1,1],
    'IOA': [1,1,0,1,1,0,1, 1,0,0,0,0,0,1, 1,1,0,0,1,1,1, 1,0,0,1,1,1,1, 1,1,1,0,0,0,1]
}


# -- Restricción de Cualificación (Roles) --
# Definimos qué puestos puede ocupar cada profesor usando Conjuntos (Sets) para búsquedas rápidas.
# P = Presidente, S = Secretario, V = Vocal.
roles_db = {
    'RRD': {'P', 'S', 'V'}, 'QYV': {'P', 'S', 'V'}, 'LHL': {'P', 'V'},
    'HLC': {'S', 'V'}, 'MSB': {'P', 'S', 'V'}, 'PMQ': {'P', 'S', 'V'},
    'QWF': {'S', 'V'}, 'EBB': {'S', 'V'}, 'IOE': {'P', 'S', 'V'}, 'IOA': {'P', 'S', 'V'}
}

# Extraemos una lista estática con los nombres de todos los profesores para iterar sobre ellos.
prof_names = list(roles_db.keys())

# ==========================================
# 2. ALGORITMO DETERMINISTA (VORAZ / GREEDY)
# ==========================================

def solver_determinista(num_tribunales=15):
    """
    Algoritmo Constructivo Voraz.
    Estrategia: First-Fit (Primer Ajuste). Recorre los tribunales secuencialmente y les
    asigna el primer hueco cronológico disponible donde se cumplan todas las restricciones.
    Al no usar funciones aleatorias, garantiza siempre el mismo resultado (Determinista).
    """
    solucion = []  # Aquí almacenaremos la configuración final de cada tribunal.


    # -- Estructura de Control para la Restricción de Solapamiento --
    # Diccionario donde la clave es el ID del slot temporal, y el valor es un conjunto de profesores.
    # Sirve para memorizar qué profesores ya están asignados a un tribunal en una hora específica.
    ocupados = {i: set() for i in range(len(slots))}

    print(f"Ejecutando asignación determinista para {num_tribunales} tribunales...")

    # Bucle Principal: construimos la solución tribunal a tribunal (de 1 a N).
    for t_id in range(1, num_tribunales + 1):
        asignado = False # Bandera para saber si ya hemos logrado ubicar este tribunal.

        # Bucle 1: recorrido cronológico de los slots de tiempo (buscando compactar calendario)
        for s_idx, slot in enumerate(slots):
            if asignado: break # Si el tribunal ya se asignó, pasamos al siguiente slot.

            # -- Aplicación de Restricciones --

            # Paso A: filtrar profesores válidos para este slot temporal.
            # Condición 1 (Disponibilidad): availability_db[p][s_idx] == 1
            # Condición 2 (No Solapamiento): p not in ocupados[s_idx]
            cands = [p for p in prof_names if availability_db[p][s_idx] == 1 and p not in ocupados[s_idx]]

            # Paso B: subdividir a los candidatos según los roles requeridos para el tribunal.
            c_P = [p for p in cands if 'P' in roles_db[p]]
            c_S = [p for p in cands if 'S' in roles_db[p]]
            c_V = [p for p in cands if 'V' in roles_db[p]]

            # Paso C: Búsqueda combinatoria local (Terna P, S, V)
            # Iteramos secuencialmente sobre las listas de candidatos (sin usar azar).
            for p in c_P:
                if asignado: break
                for s in c_S:
                    if asignado: break

                    # Condición 3 (Diferencia de Identidad): Presidente y Secretario deben ser distintos.
                    if p == s: continue

                    for v in c_V:
                        # Condición 3 (Diferencia): Vocal distinto a Presidente y Secretario
                        if p != v and s != v:


                            # ¡ÉXITO! Se han cumplido todas las restricciones para este tribunal.
                            # Guardamos la asignación (Solución Parcial).
                            solucion.append({
                                'Tribunal': t_id,
                                'Dia': slot['dia'],
                                'Hora': slot['hora'],
                                'Presidente': p,
                                'Secretario': s,
                                'Vocal': v
                            })

                            # Actualizamos el estado del sistema para el futuro:
                            # Añadimos a P, S y V a la lista de "ocupados" en esta hora concreta.
                            ocupados[s_idx].update([p, s, v])
                            asignado = True # Marcamos como completado para romper los bucles.
                            break # Rompemos el bucle de Vocales.

        # Validación de seguridad: si iteramos todos los slots y no encontramos hueco.
        if not asignado:
            print(f"Límite alcanzado: Imposible asignar el Tribunal {t_id}")

    # Transformamos la lista de diccionarios a un DataFrame para una mejor visualización.
    return pd.DataFrame(solucion)

# ==========================================
# 3. EJECUCIÓN Y VISUALIZACIÓN DE RESULTADOS
# ==========================================

# Llamamos a la función principal para 15 tribunaleS.
df_final = solver_determinista(15)

print("\n--- CALENDARIO FINAL ---")

# Imprimimos la tabla sin el índice numérico por defecto de pandas para mayor limpieza.
print(df_final.to_string(index=False))

Ejecutando asignación determinista para 15 tribunales...

--- CALENDARIO FINAL ---
 Tribunal  Dia  Hora Presidente Secretario Vocal
        1   15    15        QYV        HLC   MSB
        2   15    15        PMQ        EBB   IOE
        3   15    16        RRD        QYV   MSB
        4   15    16        PMQ        QWF   EBB
        5   15    17        RRD        QYV   LHL
        6   15    17        PMQ        HLC   QWF
        7   15    18        RRD        QYV   LHL
        8   15    18        MSB        PMQ   QWF
        9   15    18        IOE        EBB   IOA
       10   15    19        PMQ        HLC   QWF
       11   15    20        RRD        HLC   LHL
       12   15    20        MSB        QWF   IOE
       13   15    21        RRD        MSB   LHL
       14   16    15        RRD        HLC   LHL
       15   16    15        MSB        PMQ   QWF
